In [23]:
!pip install pytorch-metric-learning -q

In [ ]:
!pip install timm pytorch-metric-learning -q

import os, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from pytorch_metric_learning import losses, miners
from torch.amp import autocast, GradScaler

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# CONFIG — MAXIMUM ACCURACY
# ============================================================
class CFG:
    train_base   = "/kaggle/input/datasets/souravsheoran010/tvrid-full-train"
    test_base    = "/kaggle/input/datasets/souravsheoran010/tvrid-full-test"
    train_csv    = "/kaggle/input/datasets/souravsheoran010/tvrid-datasets/train_labels.csv"
    test_csv     = "/kaggle/input/datasets/souravsheoran010/tvrid-datasets/public_test_labels.csv"
    backbone     = "vit_base_patch16_224"   # ViT-Base — much stronger
    img_size     = 224
    embed_dim    = 768
    epochs       = 100
    warmup_epochs= 15
    batch_size   = 16                        # smaller for ViT memory
    lr           = 2e-4
    min_lr       = 1e-7
    weight_decay = 1e-4
    re_prob      = 0.4
    mixup_alpha  = 0.4                       # mixup augmentation
    cutmix_alpha = 0.4                       # cutmix augmentation
    label_smooth = 0.15
    device       = DEVICE

# ============================================================
# LOAD CSV & BUILD INDEX
# ============================================================
train_df = pd.read_csv(CFG.train_csv)
test_df  = pd.read_csv(CFG.test_csv)
print(f"Train CSV: {len(train_df)} rows | Persons: {train_df['person_id'].nunique()}")

def build_train_index(base_path):
    index = {}
    for part_dir in sorted(Path(base_path).iterdir()):
        if not part_dir.is_dir(): continue
        for person_dir in sorted(part_dir.iterdir()):
            if not person_dir.is_dir(): continue
            for cam_dir in sorted(person_dir.iterdir()):
                if not cam_dir.is_dir(): continue
                for passage_dir in sorted(cam_dir.iterdir()):
                    if not passage_dir.is_dir(): continue
                    rgb_files = sorted(passage_dir.glob("*RGB*.png"))
                    depth_map = {d.name.replace("_depth.png",""): d
                                 for d in passage_dir.glob("*depth*.png")}
                    pairs = [(str(r), str(depth_map[r.name.replace("_RGB.png","")]))
                             for r in rgb_files
                             if r.name.replace("_RGB.png","") in depth_map]
                    if pairs:
                        key = (person_dir.name, cam_dir.name, passage_dir.name)
                        index[key] = pairs
    return index

def build_test_index(base_path):
    index = {}
    for part_dir in sorted(Path(base_path).iterdir()):
        if not part_dir.is_dir(): continue
        for folder_dir in sorted(part_dir.iterdir()):
            if not folder_dir.is_dir(): continue
            rgb_files = sorted(folder_dir.glob("*RGB*.png"))
            depth_map = {d.name.replace("_depth.png",""): d
                         for d in folder_dir.glob("*depth*.png")}
            pairs = [(str(r), str(depth_map[r.name.replace("_RGB.png","")]))
                     for r in rgb_files
                     if r.name.replace("_RGB.png","") in depth_map]
            if pairs:
                index[folder_dir.name] = pairs
    return index

print("Building indexes...")
train_index = build_train_index(CFG.train_base)
test_index  = build_test_index(CFG.test_base)
print(f"Train index: {len(train_index)} | Test index: {len(test_index)}")

# ============================================================
# BUILD SAMPLES
# ============================================================
def normalize_path(p): return p.replace("\\", "/")

samples = []
for _, row in train_df.iterrows():
    parts = normalize_path(row["path"]).split("/")
    key   = (parts[0], parts[1], parts[2])
    if key in train_index:
        samples.append({
            "pid":       int(row["person_id"]),
            "cam":       int(row["cam_id"]),
            "all_pairs": train_index[key]
        })

random.shuffle(samples)
split         = int(len(samples) * 0.85)
train_samples = samples[:split]
val_samples   = samples[split:]
all_pids      = sorted(set(s["pid"] for s in samples))
pid2label     = {p: i for i, p in enumerate(all_pids)}
label2pid     = {v: k for k, v in pid2label.items()}
num_ids       = len(all_pids)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | IDs: {num_ids}")

# ============================================================
# TRANSFORMS
# ============================================================
def get_transforms(split="train"):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == "train":
        return T.Compose([
            T.Resize((CFG.img_size + 32, CFG.img_size + 32)),
            T.RandomCrop(CFG.img_size),
            T.RandomHorizontalFlip(),
            T.RandomRotation(25),
            T.ColorJitter(0.4, 0.4, 0.3, 0.1),
            T.RandomGrayscale(p=0.1),
            T.ToTensor(),
            T.Normalize(mean, std),
            T.RandomErasing(p=CFG.re_prob, scale=(0.02, 0.33))
        ])
    return T.Compose([
        T.Resize((CFG.img_size, CFG.img_size)),
        T.ToTensor(),
        T.Normalize(mean, std)
    ])

def load_depth(path):
    img = Image.open(path)
    arr = np.array(img).astype(np.float32)
    if arr.max() > 0:
        arr = (arr / arr.max() * 255).astype(np.uint8)
    else:
        arr = arr.astype(np.uint8)
    if arr.ndim == 2:
        arr = np.stack([arr]*3, axis=-1)
    elif arr.shape[2] == 4:
        arr = arr[:,:,:3]
    return Image.fromarray(arr)

# ============================================================
# MIXUP & CUTMIX
# ============================================================
def mixup_data(x1, x2, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x1.size(0), device=x1.device)
    mixed_x1 = lam * x1 + (1 - lam) * x1[idx]
    mixed_x2 = lam * x2 + (1 - lam) * x2[idx]
    y_a, y_b = y, y[idx]
    return mixed_x1, mixed_x2, y_a, y_b, lam

def cutmix_data(x1, x2, y, alpha=0.4):
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x1.size(0), device=x1.device)
    W, H = x1.size(3), x1.size(2)
    cut_rat = math.sqrt(1 - lam)
    cut_w   = int(W * cut_rat)
    cut_h   = int(H * cut_rat)
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1b = max(0, cx - cut_w//2); x2b = min(W, cx + cut_w//2)
    y1b = max(0, cy - cut_h//2); y2b = min(H, cy + cut_h//2)
    x1_new = x1.clone()
    x2_new = x2.clone()
    x1_new[:, :, y1b:y2b, x1b:x2b] = x1[idx, :, y1b:y2b, x1b:x2b]
    x2_new[:, :, y1b:y2b, x1b:x2b] = x2[idx, :, y1b:y2b, x1b:x2b]
    lam = 1 - (x2b-x1b)*(y2b-y1b)/(W*H)
    return x1_new, x2_new, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ============================================================
# DATASET
# ============================================================
class TVRIDDataset(Dataset):
    def __init__(self, samples, pid2label, split="train"):
        self.samples   = samples
        self.pid2label = pid2label
        self.split     = split
        self.tf        = get_transforms(split)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s    = self.samples[idx]
        pair = random.choice(s["all_pairs"]) if self.split == "train" \
               else s["all_pairs"][len(s["all_pairs"])//2]
        rgb  = Image.open(pair[0]).convert("RGB")
        dep  = load_depth(pair[1])
        seed = random.randint(0, 99999)
        torch.manual_seed(seed); random.seed(seed)
        rgb_t = self.tf(rgb)
        torch.manual_seed(seed); random.seed(seed)
        dep_t = self.tf(dep)
        return rgb_t, dep_t, self.pid2label[s["pid"]], s["cam"]

train_ds     = TVRIDDataset(train_samples, pid2label, "train")
val_ds       = TVRIDDataset(val_samples,   pid2label, "val")
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size,
                          shuffle=True,  num_workers=4, pin_memory=True,
                          drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size,
                          shuffle=False, num_workers=4, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ============================================================
# MODEL — ViT-Base Dual Stream
# ============================================================
class TVRIDModelViT(nn.Module):
    def __init__(self, num_ids):
        super().__init__()
        # ViT-Base for RGB
        self.rgb_enc = timm.create_model(
            "vit_base_patch16_224", pretrained=True, num_classes=0)
        # ViT-Small for Depth (lighter)
        self.dep_enc = timm.create_model(
            "vit_small_patch16_224", pretrained=True, num_classes=0)

        rgb_dim = self.rgb_enc.num_features   # 768
        dep_dim = self.dep_enc.num_features   # 384

        # Project depth to RGB dim
        self.dep_proj = nn.Sequential(
            nn.Linear(dep_dim, rgb_dim),
            nn.LayerNorm(rgb_dim)
        )

        # Cross-modal attention
        self.cross_attn = nn.MultiheadAttention(
            rgb_dim, num_heads=8, batch_first=True)
        self.cross_norm = nn.LayerNorm(rgb_dim)

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(rgb_dim * 2, CFG.embed_dim),
            nn.BatchNorm1d(CFG.embed_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )
        self.bnneck = nn.BatchNorm1d(CFG.embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.classifier = nn.Linear(CFG.embed_dim, num_ids, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.01)

    def encode(self, rgb, dep):
        f_rgb = self.rgb_enc(rgb)                        # [B, 768]
        f_dep = self.dep_enc(dep)                        # [B, 384]
        f_dep = self.dep_proj(f_dep)                     # [B, 768]

        # Cross-modal attention: RGB attends to Depth
        f_rgb_s = f_rgb.unsqueeze(1)                     # [B,1,768]
        f_dep_s = f_dep.unsqueeze(1)
        f_cross, _ = self.cross_attn(f_rgb_s, f_dep_s, f_dep_s)
        f_cross = self.cross_norm(f_cross.squeeze(1))    # [B, 768]

        # Fuse original RGB + cross-attended
        feat = self.fusion(torch.cat([f_rgb, f_cross], dim=1))
        return self.bnneck(feat)

    def forward(self, rgb, dep):
        feat = self.encode(rgb, dep)
        return feat, self.classifier(feat)

model = TVRIDModelViT(num_ids=num_ids).to(CFG.device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total/1e6:.1f}M")

# ============================================================
# LOSS & OPTIMIZER
# ============================================================
ce_loss   = nn.CrossEntropyLoss(label_smoothing=CFG.label_smooth)
trip_loss = losses.TripletMarginLoss(margin=0.3)
miner     = miners.MultiSimilarityMiner(epsilon=0.1)
scaler    = GradScaler("cuda")

optimizer = torch.optim.AdamW([
    {"params": list(model.rgb_enc.parameters()) +
               list(model.dep_enc.parameters()),    "lr": CFG.lr * 0.05},
    {"params": list(model.dep_proj.parameters()) +
               list(model.cross_attn.parameters()) +
               list(model.cross_norm.parameters()) +
               list(model.fusion.parameters()) +
               list(model.bnneck.parameters()) +
               list(model.classifier.parameters()), "lr": CFG.lr}
], weight_decay=CFG.weight_decay)

def lr_lambda(epoch):
    if epoch < CFG.warmup_epochs:
        return (epoch + 1) / CFG.warmup_epochs
    progress = (epoch - CFG.warmup_epochs) / (CFG.epochs - CFG.warmup_epochs)
    return CFG.min_lr/CFG.lr + 0.5*(1 - CFG.min_lr/CFG.lr)*(1 + math.cos(math.pi*progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ============================================================
# TRAINING LOOP WITH MIXUP + CUTMIX
# ============================================================
best_val_acc  = 0.0
best_val_loss = float("inf")
history       = {"train_loss": [], "val_loss": [], "val_acc": []}

print("\n" + "="*60)
print("STARTING MAXIMUM ACCURACY TRAINING")
print("ViT-Base + Cross-Modal Attention + MixUp + CutMix")
print("="*60)

for epoch in range(CFG.epochs):
    model.train()
    t_loss = 0.0

    for rgb, dep, label, cam in train_loader:
        rgb   = rgb.to(CFG.device)
        dep   = dep.to(CFG.device)
        label = label.to(CFG.device)

        # Apply Mixup or CutMix randomly
        r = random.random()
        if r < 0.33:
            rgb, dep, la, lb, lam = mixup_data(rgb, dep, label, CFG.mixup_alpha)
            use_mix = "mixup"
        elif r < 0.66:
            rgb, dep, la, lb, lam = cutmix_data(rgb, dep, label, CFG.cutmix_alpha)
            use_mix = "cutmix"
        else:
            la, lb, lam = label, label, 1.0
            use_mix = "none"

        optimizer.zero_grad()
        with autocast("cuda"):
            feat, logits = model(rgb, dep)
            if use_mix != "none":
                l_ce = mixup_criterion(ce_loss, logits, la, lb, lam)
            else:
                l_ce = ce_loss(logits, label)
            pairs  = miner(feat, label)
            l_trip = trip_loss(feat, label, pairs) \
                     if pairs[0].numel() > 0 \
                     else torch.tensor(0.0, device=CFG.device)
            loss   = l_ce + 0.5 * l_trip

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()

    scheduler.step()
    avg_train = t_loss / max(len(train_loader), 1)

    # Validate
    model.eval()
    v_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for rgb, dep, label, cam in val_loader:
            rgb   = rgb.to(CFG.device)
            dep   = dep.to(CFG.device)
            label = label.to(CFG.device)
            with autocast("cuda"):
                feat, logits = model(rgb, dep)
                loss         = ce_loss(logits, label)
            v_loss  += loss.item()
            correct += (logits.argmax(1) == label).sum().item()
            total   += label.size(0)

    avg_val = v_loss / max(len(val_loader), 1)
    val_acc = correct / max(total, 1)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_acc"].append(val_acc)

    if val_acc >= best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = avg_val
        torch.save(model.state_dict(), "/kaggle/working/best_vit_model.pth")

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{CFG.epochs} | "
              f"Train: {avg_train:.4f} | "
              f"Val: {avg_val:.4f} | "
              f"Acc: {val_acc*100:.1f}% | "
              f"Best: {best_val_acc*100:.1f}%")

# ============================================================
# PLOT
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,4))
ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"],   label="Val")
ax1.set_title("Loss"); ax1.legend()
ax2.plot([a*100 for a in history["val_acc"]], color="green")
ax2.set_title("Val Accuracy (%)")
plt.tight_layout()
plt.savefig("/kaggle/working/vit_training_curves.png", dpi=150)
plt.show()
print(f"\nBest Val Acc: {best_val_acc*100:.1f}%")

# ============================================================
# FINAL SUBMISSION — 5-way TTA + All Frames
# ============================================================
model.load_state_dict(torch.load("/kaggle/working/best_vit_model.pth"))
model.eval()

tta_tfs = [
    T.Compose([T.Resize((224,224)), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((256,256)), T.CenterCrop(224), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((248,248)), T.CenterCrop(224), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(p=1.0),
               T.CenterCrop(200), T.Resize((224,224)), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]

print("\nGenerating final submission with ViT + 5-way TTA...")
preds = []
for folder_name, pairs in test_index.items():
    all_logits = []
    with torch.no_grad():
        for rgb_p, dep_p in pairs:
            rgb_img = Image.open(rgb_p).convert("RGB")
            dep_img = load_depth(dep_p)
            for tf_tta in tta_tfs:
                rgb = tf_tta(rgb_img).unsqueeze(0).to(CFG.device)
                dep = tf_tta(dep_img).unsqueeze(0).to(CFG.device)
                with autocast("cuda"):
                    _, logits = model(rgb, dep)
                all_logits.append(logits.cpu())
    avg_logits = torch.stack(all_logits).mean(0)
    pred_pid   = label2pid[avg_logits.argmax(1).item()]
    preds.append({"gallery_id": folder_name, "person_id": pred_pid})

submission = pd.DataFrame(preds)
submission.to_csv("/kaggle/working/submission_vit.csv", index=False)

print(f"\n{'='*60}")
print(f"🏆 MAXIMUM ACCURACY RESULTS")
print(f"{'='*60}")
print(f"Previous best  : 94.7% (EfficientNet-B3)")
print(f"New best       : {best_val_acc*100:.1f}% (ViT-Base + CutMix + TTA)")
print(f"Submission     : /kaggle/working/submission_vit.csv")
print(f"Shape          : {submission.shape}")
print(f"\nPrediction spread:")
print(submission["person_id"].value_counts().head(10).to_string())

Device: cuda
Train CSV: 496 rows | Persons: 62
Building indexes...
Train index: 496 | Test index: 204
Train: 421 | Val: 75 | IDs: 62
Train batches: 26 | Val batches: 5
Parameters: 111.4M

STARTING MAXIMUM ACCURACY TRAINING
ViT-Base + Cross-Modal Attention + MixUp + CutMix
Epoch   5/100 | Train: 4.2051 | Val: 4.0058 | Acc: 8.0% | Best: 8.0%
Epoch  10/100 | Train: 3.8881 | Val: 3.3099 | Acc: 38.7% | Best: 38.7%
Epoch  15/100 | Train: 3.0870 | Val: 2.1679 | Acc: 76.0% | Best: 76.0%
Epoch  20/100 | Train: 2.7791 | Val: 1.5435 | Acc: 93.3% | Best: 93.3%
Epoch  25/100 | Train: 2.5164 | Val: 1.3562 | Acc: 94.7% | Best: 96.0%
Epoch  30/100 | Train: 2.3223 | Val: 1.3254 | Acc: 93.3% | Best: 96.0%
Epoch  35/100 | Train: 2.3260 | Val: 1.2843 | Acc: 94.7% | Best: 97.3%


KeyboardInterrupt: 